In [201]:
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, f1_score

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [4]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

print("Train shape:\t", data_train.shape)
print("Test shape:\t", data_test.shape)

Train shape:	 (7088, 22)
Test shape:	 (3039, 21)


In [5]:
data_train.head()
#data_test.head()

,id,CLIENTNUM,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,loan_defaulted
0,8949,708472008,44,F,3,Uneducated,Married,Less than $40K,Blue,36,...,3,6680.0,1839,4841.0,0.617,7632,95,0.532,0.275,0
1,6666,713927283,39,F,1,Graduate,Single,Unknown,Blue,34,...,1,2884.0,2517,367.0,0.693,4809,87,0.740,0.873,0
2,7120,715593783,52,M,1,Unknown,Married,$80K - $120K,Blue,36,...,2,14858.0,1594,13264.0,0.510,4286,72,0.636,0.107,0
3,2258,713237958,34,M,0,Graduate,Married,$40K - $60K,Blue,17,...,4,2638.0,2092,546.0,0.591,1868,43,0.344,0.793,0
4,3462,717569283,47,M,5,Doctorate,Single,Less than $40K,Blue,36,...,2,8896.0,1338,7558.0,0.741,4252,70,0.591,0.150,0


In [6]:
for col in data_train.columns:
    print(f"{col:<25} | {data_train[col].isna().sum()} | {data_train[col].dtype}")

id                        | 0 | int64
CLIENTNUM                 | 0 | int64
Customer_Age              | 0 | int64
Gender                    | 0 | object
Dependent_count           | 0 | int64
Education_Level           | 0 | object
Marital_Status            | 0 | object
Income_Category           | 0 | object
Card_Category             | 0 | object
Months_on_book            | 0 | int64
Total_Relationship_Count  | 0 | int64
Months_Inactive_12_mon    | 0 | int64
Contacts_Count_12_mon     | 0 | int64
Credit_Limit              | 0 | float64
Total_Revolving_Bal       | 0 | int64
Avg_Open_To_Buy           | 0 | float64
Total_Amt_Chng_Q4_Q1      | 0 | float64
Total_Trans_Amt           | 0 | int64
Total_Trans_Ct            | 0 | int64
Total_Ct_Chng_Q4_Q1       | 0 | float64
Avg_Utilization_Ratio     | 0 | float64
loan_defaulted            | 0 | int64


In [7]:
0 # delete
1 # no change
2 # devide into classes
data_process = [0, #id
                0, #CLIENTNUM
                1, #Customer_Age
                2, #Gender
                1, #Dependent_count           
                2, #Education_Level           
                2, #Marital_Status            
                2, #Income_Category           
                2, #Card_Category             
                1, #Months_on_book            
                1, #Total_Relationship_Count  
                1, #Months_Inactive_12_mon    
                1, #Contacts_Count_12_mon     
                1, #Credit_Limit              
                1, #Total_Revolving_Bal       
                1, #Avg_Open_To_Buy           
                1, #Total_Amt_Chng_Q4_Q1      
                1, #Total_Trans_Amt           
                1, #Total_Trans_Ct            
                1, #Total_Ct_Chng_Q4_Q1       
                1,] #Avg_Utilization_Ratio     

data_process_np = np.array(data_process)

In [22]:
class process_class():
    def __init__(self):
        self.counter_numeric = 0
        self.counter_category = 0
    
    def precessing(self, pandas_data, how_to_process):
        """
        input
        how_to_process - numpy array with proccessing labels [0, 1, 2]
        pandas_data - pandas array train/test array for data processing. Only features with no target column
     
        return:
        np_data_concatenate_numeric - numereic data array after processing
        np_data_concatenate_category - category data array after processing
        header - numpy array for new headers
        """
        header = np.array([])
        np_data_concatenate_numeric = np.zeros((pandas_data.shape[0], 1))
        np_data_concatenate_category = np.zeros((pandas_data.shape[0], 1))
    
        counter_numeric = 0
        counter_category = 0
        
        for col, i in zip(pandas_data.columns, range(how_to_process.shape[0])):
            print(f"i = {i}")
            if (how_to_process[i] == 0):
                # удаляем столбец. Он не нужен. А лучше сказать по другому - мы его не будет вставлять в новый numpy array
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
                print("Column not included \n")        
                continue
        
            elif (how_to_process[i] == 1):
                # копируем из pandas и вставляем в numpy array
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
                counter_numeric += 1 # для проверки
                
                np_add = pandas_data[col].fillna(0).to_numpy()[:, None] # превращаем NaN в нули
                np_data_concatenate_numeric = np.concatenate((np_data_concatenate_numeric, np_add), axis = 1)
                nan_sum = np.isnan(np_add).sum()
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num: {null_num}") # для проверки
                print(f"NaN  num: {nan_sum} \n") # для проверки. Если показывает 0 - все верно
        
                # добавляем названия в header
                header = np.append(header, col)
        
            elif (how_to_process[i] == 2):
                print(f"Column name: {col}")
                print(f"PROCESS TYPE - {how_to_process[i]}")
        
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num: {null_num}") # для проверки
                unique_data_column = pandas_data[col].unique()
                print("Unique data types", unique_data_column.shape[0])
                
                pandas_data[col] = pandas_data[col].fillna("None") # убираем NULL
                null_num = pandas_data[col].isnull().sum()
                print(f"Null num after changed: {null_num}") # для проверки 
                unique_data_column = pandas_data[col].unique()
                print("Unique data types after changed", unique_data_column.shape[0]) # для проверки 
        
                counter_category += unique_data_column.shape[0] # для проверки
                
                # добавляем в наш массив
                np_add = np.zeros((pandas_data.shape[0], unique_data_column.shape[0])) # превращаем NaN в нули
                print("shape", np_add.shape)
                
                for i_new in range(pandas_data.shape[0]):
                    for j_new in range(unique_data_column.shape[0]):
                        if (pandas_data.loc[i_new, col] == unique_data_column[j_new]):
                            np_add[i_new, j_new] = 1 # этот элемент есть
                        else:
                            continue # этого элемента нет - обнуляем
        
                np_data_concatenate_category = np.concatenate((np_data_concatenate_category, np_add), axis = 1)
        
                # добавляем названия в header
                for i in range(unique_data_column.shape[0]):
                    head = str(unique_data_column[i]) + "_" + str(col)
                    header = np.append(header, head)
        
                print("\n")
        
        np_data_concatenate_numeric = np_data_concatenate_numeric[:, :-1] # убираем первый пустой столбец, который добавили в самом начале
        np_data_concatenate_category = np_data_concatenate_category[:, :-1] 

        self.counter_numeric = counter_numeric
        self.counter_category = counter_category
        
        return np_data_concatenate_numeric, np_data_concatenate_category, header

    def concate_display(self, np_data_concatenate_numeric, np_data_concatenate_category, header):
        """
        concatenates numeric and category arrays  
        prints the data after
        """
        np_data_concatenate = np.concatenate((np_data_concatenate_numeric, np_data_concatenate_category), axis = 1)
        
        print("Численные признаки      =", np_data_concatenate_numeric.shape[1], " == ", self.counter_numeric)
        print("Категориальные признаки =", np_data_concatenate_category.shape[1], " == ", self.counter_category)
        print("Размер header           =", header.shape[0])
        print("Количество в header прав=", self.counter_numeric + self.counter_category == header.shape[0])
        print("Размер данных матрицы   =", np_data_concatenate.shape)
        print("Есть ли пропуски        =", np.isnan(np_data_concatenate).any())

        return np_data_concatenate

In [23]:
PROC = process_class()
numeric_train, category_train, header_train = PROC.precessing(data_train, data_process_np)
processed_data_np = PROC.concate_display(numeric_train, category_train, header_train)

i = 0
Column name: id
PROCESS TYPE - 0
Column not included 

i = 1
Column name: CLIENTNUM
PROCESS TYPE - 0
Column not included 

i = 2
Column name: Customer_Age
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 3
Column name: Gender
PROCESS TYPE - 2
Null num: 0
Unique data types 2
Null num after changed: 0
Unique data types after changed 2
shape (7088, 2)


i = 4
Column name: Dependent_count
PROCESS TYPE - 1
Null num: 0
NaN  num: 0 

i = 5
Column name: Education_Level
PROCESS TYPE - 2
Null num: 0
Unique data types 7
Null num after changed: 0
Unique data types after changed 7
shape (7088, 7)


i = 6
Column name: Marital_Status
PROCESS TYPE - 2
Null num: 0
Unique data types 4
Null num after changed: 0
Unique data types after changed 4
shape (7088, 4)


i = 7
Column name: Income_Category
PROCESS TYPE - 2
Null num: 0
Unique data types 6
Null num after changed: 0
Unique data types after changed 6
shape (7088, 6)


i = 8
Column name: Card_Category
PROCESS TYPE - 2
Null num: 0
Unique data types 

In [117]:
# Можно делать через OneHotEncoder. Так удобнее?
numeric_col = data_train.select_dtypes(include = ["int64", "float64"]).columns
category_col = data_train.select_dtypes(include = ["object"]).columns

#OneHot encoding
precessed_category_col = OneHotEncoder(sparse_output=False).fit_transform(data_train[category_col])
processed_numeric_col = data_train[numeric_col].drop(columns = ["loan_defaulted", "id", "CLIENTNUM"])

# Объединяем по горизонтали
processed_array = np.hstack([processed_numeric_col, precessed_category_col])
print(processed_array.shape)
print("The result is identical")

(7088, 37)
The result is identical


In [135]:
bank_train, bank_test, loan_train, loan_test = train_test_split(
    processed_array, 
    data_train["loan_defaulted"], 
    test_size=0.2, 
    stratify=data_train["loan_defaulted"], 
    random_state=42)

print("Доля дефолтов в выборке original", (data_train["loan_defaulted"] == 1).sum()/data_train.shape[0]) 
print("Доля дефолтов в выборке train \t", (loan_train == 1).sum()/loan_train.shape[0])
print("Доля дефолтов в выборке test \t", (loan_test == 1).sum()/loan_test.shape[0])

Доля дефолтов в выборке original 0.1606941309255079
Доля дефолтов в выборке train 	 0.16067019400352733
Доля дефолтов в выборке test 	 0.1607898448519041


In [136]:
#preparing pipeling for the RandomForest
Pipeline = sklearn.pipeline.Pipeline
RandomForest = sklearn.ensemble.RandomForestClassifier

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("forest", RandomForest(random_state=42))
])

In [137]:
#Grid for RandomForest
param_grid = {
    'forest__n_estimators': [100, 200, 300, 500, 1000],
    'forest__max_depth': [3, 5, 7, 10, None],
    'forest__min_samples_split': [2, 5, 10, 20, 50],
    'forest__min_samples_leaf': [5, 10, 20],
    'forest__max_features': ['sqrt', 0.1, 0.2, 0.3], 
    'forest__class_weight': ['balanced']
}

In [138]:
%%time

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(bank_train, loan_train)

print(f"Best params for GridSearch: {grid_search.best_params_}")
print(f"Best score for GridSearch: {grid_search.best_score_:.4f}")

Best params for GridSearch: {'forest__class_weight': 'balanced', 'forest__max_depth': None, 'forest__max_features': 0.3, 'forest__min_samples_leaf': 5, 'forest__min_samples_split': 2, 'forest__n_estimators': 1000}
Best score for GridSearch: 0.9866
CPU times: total: 50.5 s
Wall time: 44min 45s


In [202]:
%%time

random_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter = 30, cv = 5, scoring='f1', verbose=1, n_jobs=-1)
random_search.fit(bank_train, loan_train)

print(f"Best params for GridSearch: {grid_search.best_params_}")
print(f"Best score for GridSearch: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params for GridSearch: {'forest__class_weight': 'balanced', 'forest__max_depth': None, 'forest__max_features': 0.3, 'forest__min_samples_leaf': 5, 'forest__min_samples_split': 2, 'forest__n_estimators': 1000}
Best score for GridSearch: 0.9866
CPU times: total: 3.36 s
Wall time: 1min 10s


In [206]:
def write_txt(name, array_best_param, best_score):
    """
    writing parameters in new txt file
    """
    name_txt = name + ".txt"
    with open(name, 'w') as f:
        f.write(f"ROC AUC score {best_score:.4f}\n")
        f.write("BEST PARAM INFO\n")
        for key in array_best_param.keys():
            f.write(f"{key:<30} | {array_best_param[key]}\n")        

In [207]:
#write_txt("gridsearch_1", grid_search.best_params_, grid_search.best_score_)
write_txt("randomsearch_2", random_search.best_params_, random_search.best_score_)

In [213]:
best_params = random_search.best_params_

best_RandomForest = RandomForest(
    n_estimators=best_params['forest__n_estimators'],
    max_depth=best_params['forest__max_depth'],
    min_samples_split=best_params['forest__min_samples_split'],
    class_weight=best_params['forest__class_weight'],
    random_state=42
)

# Обучаемся на всей выборке
best_RandomForest.fit(bank_train, loan_train)

#predict
loan_predicted = best_RandomForest.predict_proba(bank_test)[:, 1]
loan_predicted = (loan_predicted >= 0.4).astype(int)

#score 
print("F1:", f1_score(loan_test, loan_predicted))

F1: 0.8879668049792531


In [214]:
data_test = pd.read_csv("test.csv")

#preparing the test sample for sending
numeric_col_test = data_test.select_dtypes(include = ["int64", "float64"]).columns
category_col_test = data_test.select_dtypes(include = ["object"]).columns

precessed_category_col_test = OneHotEncoder(sparse_output=False).fit_transform(data_test[category_col])
processed_numeric_col_test = data_test[numeric_col_test].drop(columns = ["id", "CLIENTNUM"])

# Объединяем по горизонтали
processed_array = np.hstack([processed_numeric_col_test, precessed_category_col_test])
print(processed_array.shape)
print("The result is identical 37 columns")

(3039, 37)
The result is identical 37 columns


In [227]:
#making predictiong for for sending the result
loan_predicted_test = best_RandomForest.predict_proba(processed_array)[:, 1]
loan_predicted_test = (loan_predicted_test >= 0.0).astype(int)

In [228]:
# Загружаем тестовые данные (нужны только Id)
test_ids = data_test['id']

# Создаём DataFrame для сабмита
submission = pd.DataFrame({
    'id': test_ids,
    'loan_defaulted': loan_predicted_test
})

# Проверяем формат
submission.head()

# Сохраняем в CSV (без индекса!)
submission.to_csv('submission.csv', index=False)